<a href="https://colab.research.google.com/github/danieligelnik/CCFraudProject/blob/main/Data_manipulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install dependencies as needed:
%pip install kagglehub[pandas-datasets]

In [3]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import holidays as hol
from sklearn.model_selection import train_test_split

In [4]:
def load_kagglehub_dataset(dataset_path, file_name):
  # Load the latest version
  return kagglehub.dataset_load(
      KaggleDatasetAdapter.PANDAS,
      dataset_path,
      file_name,
      # Provide any additional arguments like
      # sql_query or pandas_kwargs. See the
      # documenation for more information:
      # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
      )

In [5]:
def df_info(df, what):
  if (what == "head"):
    print("First 5 records:")
    with pd.option_context('display.max_columns', 50):
      display(df.head())
  elif (what == "columns"):
    print("Columns:")
    display(df.columns)
    print(f"Rows / columns: {df.shape}")
  elif (what == "info"):
    print("Missing values:")
    display(df.isna().sum())
    display(df.info())

# **Data**

In [27]:
# Set the path to the file you'd like to load
credit_cards_path_train = "fraudTrain.csv"
main_path="dermisfit/fraud-transactions-dataset"

# Load the latest version
df_cards_train = load_kagglehub_dataset(main_path, credit_cards_path_train)

Using Colab cache for faster access to the 'fraud-transactions-dataset' dataset.


In [28]:
df_cards_valid = train_test_split(df_cards_train, test_size=0.2, random_state=42)[1]
df_cards_valid.head(5)



,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
1045211,1045211,2020-03-09 15:09:26,577588686219,fraud_Towne LLC,misc_pos,194.51,James,Strickland,M,25454 Leonard Lake,...,40.6153,-79.4545,972,Public relations account executive,1997-10-23,fff87d4340ef756a592eac652493cf6b,1362841766,40.420453,-78.865012,0
547406,547406,2019-08-22 15:49:01,30376238035123,fraud_Friesen Ltd,health_fitness,52.32,Cynthia,Davis,F,7177 Steven Forges,...,42.8250,-124.4409,217,Retail merchandiser,1928-10-01,d0ad335af432f35578eea01d639b3621,1345650541,42.758860,-123.636337,0
110142,110142,2019-03-04 01:34:16,4658490815480264,fraud_Mohr Inc,shopping_pos,6.53,Tara,Richards,F,4879 Cristina Station,...,39.9636,-79.7853,184,Systems developer,1945-11-04,87f26e3ea33f4ff4c7a8bad2c7f48686,1330824856,40.475159,-78.898190,0
1285953,1285953,2020-06-16 20:04:38,3514897282719543,fraud_Gaylord-Powlowski,home,7.33,Steven,Faulkner,M,841 Cheryl Centers Suite 115,...,42.9580,-77.3083,10717,Cytogeneticist,1952-10-13,9c34015321c0fa2ae6fd20f9359d1d3e,1371413078,43.767506,-76.542384,0
271705,271705,2019-05-14 05:54:48,6011381817520024,"fraud_Christiansen, Goyette and Schamberger",gas_transport,64.29,Kristen,Allen,F,8619 Lisa Manors Apt. 871,...,41.6423,-104.1974,635,Product/process development scientist,1973-07-13,198437c05676f485e9be04449c664475,1336974888,41.040392,-104.092324,0


**Data encoding**

In [18]:
df_cards_train['gender'] = df_cards_train['gender'].map({'F': 1, 'M': 0})
df_cards_valid['gender'] = df_cards_valid['gender'].map({'F': 1, 'M': 0})

In [29]:
df_cards_train = pd.get_dummies(df_cards_train, columns=['category'], prefix='category', drop_first=True)
df_cards_valid = pd.get_dummies(df_cards_valid, columns=['category'], prefix='category', drop_first=True)

# The 'category' column is already replaced by get_dummies, so dropping it again is redundant and causes an error.

In [20]:
df_cards_train['trans_date_trans_time'] = pd.to_datetime(df_cards_train['trans_date_trans_time'])
df_cards_valid['trans_date_trans_time'] = pd.to_datetime(df_cards_valid['trans_date_trans_time'])


In [21]:
df_cards_train['is_weekend'] = df_cards_train['trans_date_trans_time'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)
df_cards_valid['is_weekend'] = df_cards_valid['trans_date_trans_time'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)

**Feature engineering**

In [22]:
#adding new column of age calculated from dob and trans_date_trans_time
# age at the moment of the transaction

df_cards_train['age'] = (df_cards_train['trans_date_trans_time'] - pd.to_datetime(df_cards_train.dob))
df_cards_valid['age'] = (df_cards_valid['trans_date_trans_time'] - pd.to_datetime(df_cards_valid.dob))

**Dropping columns**

In [24]:
df_cards_train = df_cards_train.drop(columns=['Unnamed: 0','cc_num','merchant','first','last','street','zip','lat','long','job','trans_num','unix_time','merch_lat','merch_long'])
df_cards_valid = df_cards_valid.drop(columns=['Unnamed: 0','cc_num','merchant','first','last','street','zip','lat','long','job','trans_num','unix_time','merch_lat','merch_long'])

**Imbalance treatment**

In [31]:
from sklearn.utils import class_weight

# Calculate class weights
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(df_cards_train['is_fraud']),
    y=df_cards_train['is_fraud']
)
class_weights_dict = dict(zip(np.unique(df_cards_train['is_fraud']), class_weights))

print("Class weights for 'is_fraud' column:")
print(class_weights_dict)
df_cards_train['is_fraud'].value_counts()


Class weights for 'is_fraud' column:
{np.int64(0): np.float64(0.5029111776656126), np.int64(1): np.float64(86.37589928057554)}


,count
is_fraud,
0,1289169
1,7506
